In [3]:
import torch

In [ ]:
CHANNEL_NAMES = [    
    "pm1",
    "pm10",
    "pm2p5[]"
]

In [ ]:
error = torch.abs(pred - y)  # [B, V, H, W]
tmp_pred = pred.clone()
tmp_pred = tmp_pred.detach().cpu().numpy()

# we only apply the freq weighted loss to the chemical variables
for var in vars:
    if var in CHANNEL_NAMES: # loop through the chemical variable string names
        bins_ind = np.digitize(tmp_pred[:, vars.index(var)], BINS[var], right=True) - 1
        bins_ind[bins_ind == len(BINS[var]) - 1] = len(BINS[var]) - 2 # incase anything is more than the last bin
        w_freq = np.zeros(len(bins_ind))
        
        # we need a bin_ind_count to get the count of the bin index. the index in COUNTS is the bin index, we need the value
        bin_ind_count = [COUNTS[var][i] for i in bins_ind.flatten()]
        bin_ind_count = np.array(bin_ind_count).reshape(bins_ind.shape)


        beta = 0.8

        # if the num of bins is 0, then the weight is 0
        mask_zero = bin_ind_count == 0
        E_num = (1 - beta ** bin_ind_count) / (1 - beta)
        w_freq = np.where(mask_zero, 0, 1 / E_num)
        
        w_freq = torch.from_numpy(w_freq).to(dtype=error.dtype, device=error.device)
        error[:, vars.index(var)] = error[:, vars.index(var)] * w_freq

# lattitude weights
w_lat = np.cos(np.deg2rad(lat))
w_lat = w_lat / w_lat.mean()  # (H, )
w_lat = torch.from_numpy(w_lat).unsqueeze(0).unsqueeze(-1).to(dtype=error.dtype, device=error.device)  # (1, H, 1)
loss_dict = {}
with torch.no_grad():
    for i, var in enumerate(vars):
        loss_dict[f"w_fmae_{var}_{log_postfix}"] = (error[:, i] * w_lat).mean()

loss_dict["w_fmae"] = np.mean([loss_dict[k].cpu() for k in loss_dict.keys()])

In [ ]:
def freq_lat_weighted_mae_val(pred, y, transform, vars, lat, log_postfix):
    """Latitude weighted mean abs error
    Args:
        y: [B, V, H, W]
        pred: [B, V, H, W]
        vars: list of variable names
        lat: H

        bins and counts are loaded from the json files that is created using the script above
    """
    


    error = torch.abs(pred - y)  # [B, V, H, W]
    tmp_pred = pred.clone()
    tmp_pred = tmp_pred.detach().cpu().numpy()

    # we only apply the freq weighted loss to the chemical variables
    for var in vars:
        if var in CHEMICAL_VARS: # loop through the chemical variable string names
            bins_ind = np.digitize(tmp_pred[:, vars.index(var)], BINS[var], right=True) - 1
            bins_ind[bins_ind == len(BINS[var]) - 1] = len(BINS[var]) - 2 # incase anything is more than the last bin
            w_freq = np.zeros(len(bins_ind))
            
            # we need a bin_ind_count to get the count of the bin index. the index in COUNTS is the bin index, we need the value
            bin_ind_count = [COUNTS[var][i] for i in bins_ind.flatten()]
            bin_ind_count = np.array(bin_ind_count).reshape(bins_ind.shape)


            beta = 0.8

            # if the num of bins is 0, then the weight is 0
            mask_zero = bin_ind_count == 0
            E_num = (1 - beta ** bin_ind_count) / (1 - beta)
            w_freq = np.where(mask_zero, 0, 1 / E_num)
            
            w_freq = torch.from_numpy(w_freq).to(dtype=error.dtype, device=error.device)
            error[:, vars.index(var)] = error[:, vars.index(var)] * w_freq

    # lattitude weights
    w_lat = np.cos(np.deg2rad(lat))
    w_lat = w_lat / w_lat.mean()  # (H, )
    w_lat = torch.from_numpy(w_lat).unsqueeze(0).unsqueeze(-1).to(dtype=error.dtype, device=error.device)  # (1, H, 1)
    loss_dict = {}
    with torch.no_grad():
        for i, var in enumerate(vars):
            loss_dict[f"w_fmae_{var}_{log_postfix}"] = (error[:, i] * w_lat).mean()

    loss_dict["w_fmae"] = np.mean([loss_dict[k].cpu() for k in loss_dict.keys()])

    return loss_dict